In [1]:
from datasets import load_dataset

klue_mrc_train = load_dataset('klue', 'mrc', split='train')
klue_mrc_train[0]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

mrc/train-00000-of-00001.parquet:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

mrc/validation-00000-of-00001.parquet:   0%|          | 0.00/8.68M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/17554 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5841 [00:00<?, ? examples/s]

{'title': '제주도 장마 시작 … 중부는 이달 말부터',
 'context': '올여름 장마가 17일 제주도에서 시작됐다. 서울 등 중부지방은 예년보다 사나흘 정도 늦은 이달 말께 장마가 시작될 전망이다.17일 기상청에 따르면 제주도 남쪽 먼바다에 있는 장마전선의 영향으로 이날 제주도 산간 및 내륙지역에 호우주의보가 내려지면서 곳곳에 100㎜에 육박하는 많은 비가 내렸다. 제주의 장마는 평년보다 2~3일, 지난해보다는 하루 일찍 시작됐다. 장마는 고온다습한 북태평양 기단과 한랭 습윤한 오호츠크해 기단이 만나 형성되는 장마전선에서 내리는 비를 뜻한다.장마전선은 18일 제주도 먼 남쪽 해상으로 내려갔다가 20일께 다시 북상해 전남 남해안까지 영향을 줄 것으로 보인다. 이에 따라 20~21일 남부지방에도 예년보다 사흘 정도 장마가 일찍 찾아올 전망이다. 그러나 장마전선을 밀어올리는 북태평양 고기압 세력이 약해 서울 등 중부지방은 평년보다 사나흘가량 늦은 이달 말부터 장마가 시작될 것이라는 게 기상청의 설명이다. 장마전선은 이후 한 달가량 한반도 중남부를 오르내리며 곳곳에 비를 뿌릴 전망이다. 최근 30년간 평균치에 따르면 중부지방의 장마 시작일은 6월24~25일이었으며 장마기간은 32일, 강수일수는 17.2일이었다.기상청은 올해 장마기간의 평균 강수량이 350~400㎜로 평년과 비슷하거나 적을 것으로 내다봤다. 브라질 월드컵 한국과 러시아의 경기가 열리는 18일 오전 서울은 대체로 구름이 많이 끼지만 비는 오지 않을 것으로 예상돼 거리 응원에는 지장이 없을 전망이다.',
 'news_category': '종합',
 'source': 'hankyung',
 'guid': 'klue-mrc-v1_train_12759',
 'is_impossible': False,
 'question_type': 1,
 'question': '북태평양 기단과 오호츠크해 기단이 만나 국내에 머무르는 기간은?',
 'answers': {'answer_start': [478, 478]

In [2]:
from sentence_transformers import SentenceTransformer

# 임베딩 모델
sentence_model = SentenceTransformer('shangrilar/klue-roberta-base-klue-sts')

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
from datasets import load_dataset

klue_mrc_train = load_dataset('klue', 'mrc', split='train')
klue_mrc_test = load_dataset('klue', 'mrc', split='validation')

df_train = klue_mrc_train.to_pandas()
df_test = klue_mrc_test.to_pandas()

df_train = df_train[['title', 'question', 'context']]
df_test = df_test[['title', 'question', 'context']]

In [4]:
def add_ir_context(df):
    irrelevant_contexts = []
    for idx, row in df.iterrows():
        title = row['title']
        irrelevant_contexts.append(df.query(f"title != '{title}'").sample(n=1)['context'].values[0])
    df['irrelevant_context'] = irrelevant_contexts
    return df

df_train_ir = add_ir_context(df_train)
df_test_ir = add_ir_context(df_test)

In [5]:
# 성능 평가 데이터 생성.
from sentence_transformers import InputExample

examples = []
for idx, row in df_test_ir.iterrows():
    examples.append(InputExample(texts=[row['question'], row['context']], label=1)) # question, context 컬럼은 서로 관련 있음.
    examples.append(InputExample(texts=[row['question'], row['irrelevant_context']], label=0)) # 서로 관련 없음.

In [6]:
# 임베딩 모델 성능 평가.
from sentence_transformers.evaluation import EmbeddingSimilarityEvaluator

evaluator = EmbeddingSimilarityEvaluator.from_input_examples(examples)
evaluator(sentence_model)

{'pearson_cosine': 0.8062475028469877, 'spearman_cosine': 0.8159414914878675}

In [7]:
# MNR(Multiple Negatives Ranking) 손실 함수를 사용해 모델을 미세 조정.

train_samples = []
for idx, row in df_train_ir.iterrows():
    train_samples.append(InputExample(texts=[row['question'], row['context']]))


from sentence_transformers import datasets
batch_size = 16
loader = datasets.NoDuplicatesDataLoader(train_samples, batch_size=batch_size) # 중복 제거.

from sentence_transformers import losses
loss = losses.MultipleNegativesRankingLoss(sentence_model)

epochs = 1
save_path = './klue_mrc_mnr'

# 미세 조정
sentence_model.fit(train_objectives=[(loader, loss)],
                  epochs=epochs,
                  warmup_steps=100,
                  output_path=save_path,
                  show_progress_bar=True)

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss
500,0.164159
1000,0.109477


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
# 임베딩 모델 성능 평가.
evaluator(sentence_model)

{'pearson_cosine': 0.9124715012483472, 'spearman_cosine': 0.8599520340275971}

In [9]:
# 교차 인코더로 사용할 사전 학습 모델 불러오기.
from sentence_transformers.cross_encoder import CrossEncoder

cross_model = CrossEncoder('klue/roberta-small', num_labels=1) # 많은 계산을 하기 때문에 파라미터 수가 작은 모델 사용.

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/273M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: klue/roberta-small
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/971 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [10]:
# 교차 인코더 성능 평가.
from sentence_transformers.cross_encoder.evaluation import CECorrelationEvaluator

ce_evaluator = CECorrelationEvaluator.from_input_examples(examples)
ce_evaluator(cross_model)

{'pearson': -0.031852676917126524, 'spearman': -0.029366946973962183}

In [11]:
# 교차 인코더 학습 데이터셋 준비.
train_samples = []
for idx, row in df_train_ir.iterrows():
    train_samples.append(InputExample(texts=[row['question'], row['context']], label=1))
    train_samples.append(InputExample(texts=[row['question'], row['irrelevant_context']], label=0))

In [12]:
# 교차 인코더 학습.
from torch.utils.data import DataLoader

train_batch_size = 16
num_epochs = 1
model_save_path = 'output/training_mrc'

train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=train_batch_size)
cross_model.fit(train_dataloader=train_dataloader,
                epochs=num_epochs,
                warmup_steps=100,
                output_path=model_save_path)

Token indices sequence length is longer than the specified maximum sequence length for this model (952 > 512). Running this sequence through the model will result in indexing errors


Step,Training Loss
500,0.246997
1000,0.079214
1500,0.060962
2000,0.050425


In [13]:
ce_evaluator(cross_model)

{'pearson': 0.9804514113638543, 'spearman': 0.8642944100597255}

In [14]:
from datasets import load_dataset

klue_mrc_test = load_dataset('klue', 'mrc', split='validation')
klue_mrc_test = klue_mrc_test.train_test_split(test_size=1000, seed=42)['test']

In [15]:
!pip install -qqq faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.2 MB/s eta 0:00:00


In [16]:
import faiss

# 검증 데이터셋 -> 문장 임베딩 변환 -> faiss 인덱스에 저장.
def make_embedding_index(sentence_model, corpus):
    embeddings = sentence_model.encode(corpus)
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index

# 검색 쿼리와 유사한 k개 문서 검색.
def find_embedding_top_k(query, sentence_model, index, k):
    embeddings = sentence_model.encode([query])
    distances, indices = index.search(embeddings, k)
    return indices

In [28]:
import time
import numpy as np

def make_question_context_pairs(question_idx, indices):
    question = klue_mrc_test['question'][int(question_idx)]
    return [[question, klue_mrc_test['context'][int(idx)]] for idx in indices]

# 교차 인코더의 유사도 점수 계산 결과에 따라 순위를 재정렬.
def rerank_top_k(cross_model, question_idx, indices, k):
    indices = np.array(indices, dtype=int)
    input_examples = make_question_context_pairs(question_idx, indices)
    relevance_scores = cross_model.predict(input_examples)
    reranked_indices = indices[np.argsort(relevance_scores)[::-1]] # 유사도 점수가 높은 순으로 인덱스 재정렬.
    return reranked_indices[:k]

# 적중률(성능 평가)
def evaluate_hit_rate(datasets, embedding_model, index, k=10):
    start_time = time.time()
    predictions = []

    # 각 질문별로 k개의 유사 문서 검색.
    for question in datasets['question']:
        topk = find_embedding_top_k(question, embedding_model, index, k)[0]
        predictions.append([int(x) for x in topk])

    # 검색 결과 내에 정답 데이터가 포함돼 있는 경우 정답을 맞췄다고 계산.
    total_prediction_count = len(predictions)
    hit_count = 0
    contexts = datasets['context']

    for idx, prediction in enumerate(predictions):
        for pred in prediction:
            if contexts[int(pred)] == contexts[int(idx)]:
                hit_count += 1
                break

    end_time = time.time()
    return hit_count / total_prediction_count, end_time - start_time

In [29]:
# 임베딩 모델 평가: 적중률, 걸린 시간 반환.
from sentence_transformers import SentenceTransformer

# 1. 기본 임베딩 모델
base_embedding_model = SentenceTransformer('shangrilar/klue-roberta-base-klue-sts')
base_index = make_embedding_index(base_embedding_model, klue_mrc_test['context']) # 벡터 검색에 사용할 인덱스 생성.
evaluate_hit_rate(klue_mrc_test, base_embedding_model, base_index, 10)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

(0.873, 19.977954864501953)

In [30]:
# 2. 미세 조정한 임베딩 모델
finetuned_embedding_model = SentenceTransformer('shangrilar/klue-roberta-base-klue-sts-mrc')
finetuned_index = make_embedding_index(finetuned_embedding_model, klue_mrc_test['context'])
evaluate_hit_rate(klue_mrc_test, finetuned_embedding_model, finetuned_index, 10)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/971 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

(0.952, 13.845140933990479)

In [33]:
# 3. 미세 조정한 임베딩 모델 + 교차 인코더
import time
import numpy as np
from tqdm.auto import tqdm

def evaluate_hit_rate_with_rerank(
    datasets,
    embedding_model,
    cross_model,
    index,
    bi_k=30,      # 추출할 상위 문서수.
    cross_k=10
):
    start_time = time.time()
    predictions = []

    for question_idx, question in enumerate(tqdm(datasets['question'])):
        indices = find_embedding_top_k(question, embedding_model, index, bi_k)[0]
        indices = [int(i) for i in indices]
        reranked = rerank_top_k(cross_model, question_idx, indices, k=cross_k) # 순위 재정렬.
        predictions.append([int(i) for i in reranked])

    total_prediction_count = len(predictions)
    hit_count = 0
    contexts = datasets['context']

    for idx, prediction in enumerate(predictions):
        for pred in prediction:
            if contexts[int(pred)] == contexts[int(idx)]:
                hit_count += 1
                break

    end_time = time.time()
    return hit_count / total_prediction_count, end_time - start_time, predictions

In [34]:
hit_rate, cosumed_time, predictions = evaluate_hit_rate_with_rerank(klue_mrc_test, finetuned_embedding_model, cross_model, finetuned_index, bi_k=30, cross_k=10)
hit_rate, cosumed_time # 히트율은 제일 높지만, 교차 인코더 때문에 시간 증가.

  0%|          | 0/1000 [00:00<?, ?it/s]

(0.976, 548.643072605133)